# ROGII Wellbore Geology - Conv1D Inference

Load the trained Conv1D model, predict TVT for test wells.

**Runtime**: Kaggle GPU - **Author**: Samir Attrah

In [1]:
# Cell 1: Environment & Imports
import glob
import os
import pickle
import warnings

# Fix: Use KERAS_BACKEND=jax with safe pre-allocation instead of 'platform' allocator.
# XLA_PYTHON_CLIENT_ALLOCATOR='platform' releases GPU memory back to CUDA using pointers
# it no longer owns, causing CUDA_ERROR_INVALID_VALUE on Kaggle T4 GPUs.
# XLA_PYTHON_CLIENT_MEM_FRACTION=0.7 pre-allocates 70% of VRAM safely, leaving room
# for the Keras model weights without fragmenting the CUDA context.
os.environ["KERAS_BACKEND"] = "jax"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "true"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.7"

import jax

# Fix: Do NOT enable x64 on GPU. float64 doubles VRAM usage and causes CUDA context
# conflicts when JAX and Keras/XLA share the same device. Inference stays in float32.
# jax.config.update('jax_enable_x64', True)  # Disabled: unsafe on Kaggle GPU

import jax.numpy as jnp
import keras
import numpy as np
import polars as pl

warnings.filterwarnings("ignore")
print(f"Keras: {keras.__version__}, Backend: {keras.backend.backend()}")


Keras: 3.10.0, Backend: jax


In [2]:
# Cell 2: Auto-detect dataset path
def find_data_dir():
    candidates = [
        "/kaggle/input/competitions/rogii-wellbore-geology-prediction",
        "/kaggle/input/rogii-wellbore-geology-prediction",
        "/home/samer/Documents/competitions/ROGII/dataset",
    ]
    for path in candidates:
        if os.path.isdir(path):
            if "test" in os.listdir(path):
                return path
    raise FileNotFoundError("Dataset not found.")


DATA_DIR = find_data_dir()


In [3]:
# Cell 3: Auto-detect model artifact
def find_model_dir():
    candidates = [
        "/home/samer/Documents/competitions/ROGII/outputs",
        "/kaggle/working",
        "/kaggle/input/models/samerattrah/rogii-21-05/keras/default/19",
    ]
    for path in candidates:
        model_path = os.path.join(path, "optimized_dense_model_augmented.keras")
        scaler_path = os.path.join(path, "dense_opt_scaler_params.pkl")
        if (
            os.path.isdir(path)
            and os.path.exists(model_path)
            and os.path.exists(scaler_path)
        ):
            return path
    raise FileNotFoundError(
        "Could not find optimized dense model and scaler artifacts."
    )


MODEL_DIR = find_model_dir()
print(f"Using model artifacts from: {MODEL_DIR}")


Using model artifacts from: /kaggle/input/models/samerattrah/rogii-21-05/keras/default/19


In [4]:
# Cell 4: Configuration
import os

# Explicitly defined data paths
DATA_DIR = "/home/samer/Documents/competitions/ROGII/dataset"
TEST_DATA_DIR = os.path.join(DATA_DIR, "test")

OUT_DIR = (
    "/kaggle/working"
    if os.path.isdir("/kaggle")
    else "/home/samer/Documents/competitions/ROGII/outputs"
)
os.makedirs(OUT_DIR, exist_ok=True)

MODEL_DIR = "/home/samer/Documents/competitions/ROGII/outputs"

CONFIG = {
    "data_dir": DATA_DIR,
    "test_data_dir": TEST_DATA_DIR,
    # Primary model file: optimized_dense_model_augmented.keras (Conv1D)
    "model_path": os.path.join(MODEL_DIR, "optimized_dense_model_augmented.keras"),
    "scaler_path": os.path.join(MODEL_DIR, "dense_opt_scaler_params.pkl"),
    "submission_path": os.path.join(OUT_DIR, "submission.csv"),
}

FEATURE_COLS = ["MD", "X", "Y", "Z", "GR", "TVT_input"]
print(f"Data Dir: {DATA_DIR}")
print(f"Test Data Dir: {TEST_DATA_DIR}")
print(f"Model Path: {CONFIG['model_path']}")


In [5]:
# Cell 5: Load model & scaler
import gc

print("Loading model...")
model = keras.saving.load_model(CONFIG["model_path"])
print(f"Model loaded. Input shape: {model.input_shape}")

print("Loading scaler...")
with open(CONFIG["scaler_path"], "rb") as f:
    scaler = pickle.load(f)
print("Scaler loaded.")

gc.collect()


In [6]:
# Cell 6: Prediction Logic
import gc

def preprocess(df, feature_cols):
    """Preprocess one well with inference-safe handling of missing features."""
    for col in feature_cols:
        if col in df.columns:
            df = df.with_columns(
                pl.col(col)
                .interpolate()
                .fill_null(strategy="forward")
                .fill_null(strategy="backward")
                .fill_null(0.0)
            )
    return df


def get_submission_index(sample_sub):
    """Adds well_id and zero-based row_idx parsed from sample_submission ids."""
    # Corrected regex to match ID format: well_id_row_idx
    return sample_sub.with_columns(
        [
            pl.col("id").str.extract(r"^(.+)_(\d+)$", 1).alias("well_id"),
            pl.col("id")
            .str.extract(r"^(.+)_(\d+)$", 2)
            .cast(pl.Int64)
            .alias("row_idx"),
        ]
    )


def normalize_features(df, scaler):
    """Applies the exact scaler saved during training.

    Uses float32 throughout to avoid CUDA_ERROR_INVALID_VALUE caused by
    float64 operations conflicting with the GPU CUDA context on Kaggle T4 GPUs.
    JAX float64 on GPU requires jax_enable_x64, which conflicts with the
    Keras/XLA shared CUDA context.
    """
    feature_cols = scaler.get("feature_cols", FEATURE_COLS)
    raw_feats = df.select(feature_cols).to_numpy()
    # Fix: Use float32 instead of float64 to stay GPU-safe without jax_enable_x64.
    feats = jnp.array(raw_feats, dtype=jnp.float32)
    mean = jnp.array(scaler["feat_mean"], dtype=jnp.float32)
    std = jnp.array(scaler["feat_std"], dtype=jnp.float32)
    std = jnp.where(std == 0, 1.0, std)
    return (feats - mean) / std


def denormalize_target(yn, scaler):
    """Converts normalized model outputs back to raw TVT units (float32)."""
    target_mean = jnp.array(scaler["target_mean"], dtype=jnp.float32)
    target_std = jnp.array(scaler["target_std"], dtype=jnp.float32)
    return (jnp.array(yn, dtype=jnp.float32) * target_std) + target_mean


def smooth_prediction_zone(preds, row_idxs, window=11, anchor_weight=0.15):
    """Smooths only submitted rows and softly anchors the first row to continuity."""
    if len(row_idxs) == 0:
        return preds

    preds = preds.copy()
    zone = np.asarray(row_idxs, dtype=np.int64)
    zone_vals = preds[zone]

    if len(zone_vals) >= window:
        kernel = np.ones(window, dtype=np.float64) / window
        pad_left = window // 2
        pad_right = window - 1 - pad_left
        padded = np.pad(zone_vals, (pad_left, pad_right), mode="edge")
        zone_vals = np.convolve(padded, kernel, mode="valid")

    first_idx = int(zone[0])
    if first_idx > 0:
        zone_vals[0] = (1.0 - anchor_weight) * zone_vals[0] + anchor_weight * preds[
            first_idx - 1
        ]

    preds[zone] = zone_vals
    return preds


def predict_well(model, df, scaler, row_idxs=None, postprocess=True):
    """Inference for one well using the convolutional model."""
    feature_cols = scaler.get("feature_cols", FEATURE_COLS)
    df = preprocess(df, feature_cols)
    feats_n = normalize_features(df, scaler)
    
    # Detect window size from model input shape (None, window_size, features)
    input_shape = model.input_shape
    ws = input_shape[1] if input_shape[1] is not None else 1
    n = len(feats_n)
    
    if ws == 1:
        # Simple reshape for pointwise Conv1D or Dense-like Conv1D
        X = np.array(feats_n, dtype=np.float32).reshape(-1, 1, len(feature_cols))
        yn = model.predict(X, batch_size=1024, verbose=0).ravel()
    else:
        # Sliding window for sequence-based Conv1D/LSTM
        starts = range(0, n - ws + 1)
        X = np.stack([np.array(feats_n[s : s + ws], dtype=np.float32) for s in starts])
        yn_seq = model.predict(X, batch_size=1024, verbose=0).ravel()
        
        # Fill prefix for rows that couldn't be the end of a full window
        yn = np.zeros(n, dtype=np.float32)
        yn[ws - 1 :] = yn_seq
        yn[: ws - 1] = yn_seq[0]

    # Cast back to float64 numpy only after JAX ops are done (safe host-side cast)
    yp = np.array(denormalize_target(yn, scaler), dtype=np.float64)

    if postprocess and row_idxs is not None:
        yp = smooth_prediction_zone(yp, row_idxs)

    return yp


In [ ]:
# Cell 7: Run Inference
sample_sub = get_submission_index(
    pl.read_csv(os.path.join(CONFIG["data_dir"], "sample_submission.csv"))
)

# Debug: check for None values in indices
agg_sub = (
    sample_sub.group_by("well_id")
    .agg(pl.min("row_idx").alias("min_idx"), pl.max("row_idx").alias("max_idx"))
)

# Filter out None values to prevent TypeError
submission_windows = {
    row["well_id"]: list(range(row["min_idx"], row["max_idx"] + 1))
    for row in agg_sub.filter(
        pl.col("min_idx").is_not_null() & pl.col("max_idx").is_not_null()
    ).iter_rows(named=True)
}
test_ids = sorted(submission_windows)

all_preds = {}
for wid in test_ids:
    print(f"Predicting {wid}...")
    well_path = os.path.join(CONFIG["test_data_dir"], f"{wid}__horizontal_well.csv")
    if not os.path.exists(well_path):
        print(f"  Warning: File not found for {wid}")
        continue
    
    try:
        df = pl.read_csv(well_path, infer_schema_length=10000)
        row_idxs = submission_windows[wid]
        all_preds[wid] = predict_well(model, df, scaler, row_idxs=row_idxs)
    except Exception as e:
        print(f"  Error predicting {wid}: {e}")

rows = []
missing = 0
for r in sample_sub.iter_rows(named=True):
    arr = all_preds.get(r["well_id"])
    if arr is None or r["row_idx"] is None or r["row_idx"] >= len(arr):
        val = 0.0
        missing += 1
    else:
        val = float(arr[r["row_idx"]])
    rows.append({"id": r["id"], "tvt": val})

if missing:
    print(f"Warning: {missing} submission rows were missing predictions and were filled with 0.0")

submission = pl.DataFrame(rows)
submission.write_csv(CONFIG["submission_path"])
print(f"Submission saved to {CONFIG['submission_path']}")
print(
    submission.select(
        pl.col("tvt").min().alias("min"),
        pl.col("tvt").max().alias("max"),
        pl.col("tvt").mean().alias("mean"),
    )
)


In [ ]:
# Cell 8: Visualise Predictions
import matplotlib.pyplot as plt

ANALYTICS_DIR = "../analytics"
os.makedirs(ANALYTICS_DIR, exist_ok=True)


def plot_predictions(all_preds, test_ids, submission_windows, max_plots=5):
    # Limit plots to avoid kernel crash during rendering
    plot_ids = test_ids[:max_plots]
    n_wells = len(plot_ids)
    if n_wells == 0:
        return

    fig, axes = plt.subplots(n_wells, 1, figsize=(15, 4 * n_wells), sharex=False)
    if n_wells == 1:
        axes = [axes]

    for i, wid in enumerate(plot_ids):
        preds = all_preds[wid]
        axes[i].plot(preds, label="Predicted TVT", color="blue", lw=1)
        rows = submission_windows.get(wid, [])
        if rows:
            axes[i].axvspan(
                rows[0], rows[-1], color="orange", alpha=0.15, label="Submission rows"
            )
        axes[i].set_title(f"Well {wid} - Predicted TVT (Conv1D)")
        axes[i].set_ylabel("TVT")
        axes[i].legend()
        axes[i].grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(ANALYTICS_DIR, "test_predictions_plots_conv1d.png"))
    plt.show()


if "all_preds" in locals() and len(all_preds) > 0:
    plot_predictions(all_preds, test_ids, submission_windows)
